# PROCESADO DIVIPOLA FINAL

## 1. LIBRERÍAS Y RUTAS

In [3]:
import numpy as np
import pandas as pd

# ── Rutas ─────────────────────────────────────────────────────────────────────
RUTA_ENTRADA = (
    "/kaggle/input/datasets/nicolasacostaa/divipola-actualizada-2025/"
    "Listados_DIVIPOLA.xlsx"
)
HOJA         = "Cabeceras - Centros Poblados"
RUTA_SALIDA  = "/kaggle/working/divipola_cabeceras_centros_poblados.parquet"

# ── Mapeo de tipo de área geográfica ─────────────────────────────────────────
MAPA_TIPO = {
    "CM": "Cabecera Municipal",
    "CP": "Centros Poblados y Rural Disperso",
}

# ── Nombres de columnas en el orden final del parquet ─────────────────────────
COLUMNAS_SALIDA = [
    "COD_DPTO",
    "NOM_DPTO",
    "COD_MPIO",
    "NOM_MPIO",
    "COD_CP",
    "NOM_CP",
    "ÁREA_GEOGRÁFICA",
    "LONGITUD",
    "LATITUD",
    "GEOMETRIA_PUNTO_WKT",
]

## 2. FUNCIONES

### 2.1. Cargar Hoja

In [4]:
def cargar_hoja(ruta: str, hoja: str) -> pd.DataFrame:
    """
    Carga la hoja 'Cabeceras - Centros Poblados' del archivo DIVIPOLA.

    El archivo tiene 10 filas de encabezado/título antes de los datos
    reales, por lo que se usa header=10 para saltar directamente a la
    fila de nombres de columnas.

    Los encabezados originales tienen nombres duplicados con espacios
    (Código, Nombre × 3 niveles: dpto, mpio, centro poblado), por lo
    que se renombran inmediatamente a nombres internos sin ambigüedad.

    Parameters
    ----------
    ruta : str   Ruta al archivo .xlsx.
    hoja : str   Nombre de la hoja a leer.

    Returns
    -------
    pd.DataFrame
        DataFrame crudo con nombres de columna internos sin ambigüedad.
    """
    print(f"[INFO] Cargando hoja '{hoja}' desde: {ruta}")
    df = pd.read_excel(ruta, sheet_name=hoja, header=10)

    # Renombrar columnas a nombres internos (los originales son ambiguos)
    df.columns = [
        "_cod_dpto", "_nom_dpto",
        "_cod_mpio", "_nom_mpio",
        "_cod_cp",   "_nom_cp",
        "_tipo",
        "_longitud", "_latitud",
        "_nota",
    ]
    print(f"  → Filas totales (con encabezado y notas): {df.shape[0]:,}")
    return df

### 2.2. Filtrar Filas Válidas

In [5]:
def filtrar_filas_validas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Retiene únicamente las filas con datos reales, descartando:
      - Filas completamente vacías (NaN en toda la fila).
      - Filas de notas al pie (el archivo termina con textos legales
        que aparecen en la columna de código de departamento).

    El criterio de fila válida es que las tres claves principales
    (_cod_dpto, _cod_mpio, _cod_cp) sean no nulas y que _cod_dpto
    sea un código de 2 caracteres (no un texto de nota).

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """
    print("[INFO] Filtrando filas válidas...")
    mascara = (
        df["_cod_dpto"].notna() &
        df["_cod_mpio"].notna() &
        df["_cod_cp"].notna() &
        df["_cod_dpto"].astype(str).str.strip().str.len().le(2)  # excluye notas
    )
    df_val = df[mascara].copy().reset_index(drop=True)
    print(f"  → Filas válidas : {len(df_val):,}")
    print(f"  → Filas descartadas (vacías / notas): {len(df) - len(df_val):,}")
    return df_val

### 2.3. Procesar Códigos

In [6]:
def procesar_codigos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza los códigos DIVIPOLA al formato string con cero a la izquierda
    para garantizar compatibilidad con el dataset de defunciones.

    COD_DPTO  → string de 2 dígitos.  Ejemplo: '5'  → '05'
    COD_MPIO  → string de 5 dígitos.  Ejemplo: 5001.0 → '05001'
                El campo original es un float porque Excel lo almacena
                como número; se convierte a int antes de formatear.
    COD_CP    → string de 8 dígitos.  Ejemplo: 5001000.0 → '05001000'
                Mismo caso que COD_MPIO.

    Compatibilidad cruzada:
      - COD_DPTO cruza con 'CÓDIGO DEPARTAMENTO' en defunciones.
      - COD_MPIO cruza con 'cod_muni_def' en defunciones.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """
    print("[INFO] Procesando códigos DIVIPOLA...")
    df = df.copy()

    df["COD_DPTO"] = (
        df["_cod_dpto"].astype(str).str.strip().str.zfill(2)
    )
    df["COD_MPIO"] = (
        df["_cod_mpio"].astype(float).astype(int).astype(str).str.zfill(5)
    )
    df["COD_CP"] = (
        df["_cod_cp"].astype(float).astype(int).astype(str).str.zfill(8)
    )

    print(f"  → COD_DPTO muestra : {df['COD_DPTO'].head(3).tolist()}")
    print(f"  → COD_MPIO muestra : {df['COD_MPIO'].head(3).tolist()}")
    print(f"  → COD_CP   muestra : {df['COD_CP'].head(3).tolist()}")
    print(f"  → Dptos únicos     : {df['COD_DPTO'].nunique()}")
    print(f"  → Mpios únicos     : {df['COD_MPIO'].nunique():,}")
    print(f"  → CPs únicos       : {df['COD_CP'].nunique():,}")
    return df

### 2.4. Procesar Nombres

In [7]:
def procesar_nombres(df: pd.DataFrame) -> pd.DataFrame:
    """
    Limpia los campos de nombre: strip de espacios y conversión a mayúsculas.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """
    print("[INFO] Limpiando campos de nombre...")
    df = df.copy()

    for origen, destino in [
        ("_nom_dpto", "NOM_DPTO"),
        ("_nom_mpio", "NOM_MPIO"),
        ("_nom_cp",   "NOM_CP"),
    ]:
        df[destino] = df[origen].astype(str).str.strip().str.upper()

    return df

### 2.5. Procesar Área Geográfica

In [8]:
def procesar_area_geografica(df: pd.DataFrame,
                              mapa: dict) -> pd.DataFrame:
    """
    Reemplaza los códigos cortos del campo Tipo por etiquetas descriptivas
    y renombra el campo a ÁREA_GEOGRÁFICA.

    Mapeo aplicado:
      CM  →  Cabecera Municipal
      CP  →  Centros Poblados y Rural Disperso

    Cualquier valor no contemplado en el mapa se conserva tal como viene
    (comportamiento seguro ante futuras actualizaciones del archivo).

    Parameters
    ----------
    df   : pd.DataFrame
    mapa : dict   Diccionario código → etiqueta.

    Returns
    -------
    pd.DataFrame
    """
    print("[INFO] Procesando campo ÁREA_GEOGRÁFICA...")
    df = df.copy()

    df["ÁREA_GEOGRÁFICA"] = (
        df["_tipo"]
        .astype(str)
        .str.strip()
        .map(mapa)
        .fillna(df["_tipo"].astype(str).str.strip())  # conservar si no está en mapa
    )

    print("  → Distribución ÁREA_GEOGRÁFICA:")
    for etiqueta, conteo in df["ÁREA_GEOGRÁFICA"].value_counts().items():
        print(f"      {etiqueta:<40} {conteo:>6,}")
    return df

### 2.6. Procesar Coordenadas y Geometria

In [9]:
def procesar_coordenadas_y_geometria(df: pd.DataFrame) -> pd.DataFrame:
    """
    Valida las coordenadas y genera el campo GEOMETRIA_PUNTO_WKT.

    LONGITUD y LATITUD vienen como float64 directamente desde Excel,
    por lo que no requieren conversión adicional.

    GEOMETRIA_PUNTO_WKT
    ────────────────────
    Representación geográfica del punto en formato WKT (Well-Known Text),
    estándar OGC compatible con GeoPandas, PostGIS, QGIS y otras
    herramientas GIS.
    Formato: 'POINT (longitud latitud)'
    Ejemplo: 'POINT (-75.581775 6.246631)'

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
    """
    print("[INFO] Procesando coordenadas y generando GEOMETRIA_PUNTO_WKT...")
    df = df.copy()

    df["LONGITUD"] = pd.to_numeric(df["_longitud"], errors="coerce")
    df["LATITUD"]  = pd.to_numeric(df["_latitud"],  errors="coerce")

    n_nulos_lon = df["LONGITUD"].isna().sum()
    n_nulos_lat = df["LATITUD"].isna().sum()
    print(f"  → LONGITUD: [{df['LONGITUD'].min():.6f}, {df['LONGITUD'].max():.6f}]  "
          f"| nulos: {n_nulos_lon}")
    print(f"  → LATITUD : [{df['LATITUD'].min():.6f}, {df['LATITUD'].max():.6f}]  "
          f"| nulos: {n_nulos_lat}")

    # Generar WKT solo donde ambas coordenadas son válidas
    tiene_coords = df["LONGITUD"].notna() & df["LATITUD"].notna()
    df["GEOMETRIA_PUNTO_WKT"] = None
    df.loc[tiene_coords, "GEOMETRIA_PUNTO_WKT"] = (
        "POINT (" +
        df.loc[tiene_coords, "LONGITUD"].round(6).astype(str) + " " +
        df.loc[tiene_coords, "LATITUD"].round(6).astype(str) +
        ")"
    )

    print(f"  → WKT generados : {tiene_coords.sum():,}")
    print(f"  → WKT muestra   : {df['GEOMETRIA_PUNTO_WKT'].iloc[0]}")
    return df

### 2.7. Seleccionar y Ordenar

In [10]:
def seleccionar_y_ordenar(df: pd.DataFrame,
                           columnas: list) -> pd.DataFrame:
    """
    Selecciona únicamente las columnas del parquet de salida en el
    orden definido por COLUMNAS_SALIDA, descartando las columnas
    internas prefijadas con '_'.

    Parameters
    ----------
    df       : pd.DataFrame
    columnas : list   Lista ordenada de nombres de columna finales.

    Returns
    -------
    pd.DataFrame
    """
    print("[INFO] Seleccionando y ordenando columnas finales...")
    df_out = df[columnas].copy()
    print(f"  → Columnas : {df_out.columns.tolist()}")
    print(f"  → Shape    : {df_out.shape[0]:,} filas × {df_out.shape[1]} columnas")
    return df_out

### 2.8. Diagnosticar

In [11]:
def diagnosticar(df: pd.DataFrame) -> None:
    """
    Imprime un resumen de calidad del DataFrame final:
      - Tipos de datos y conteo de nulos por columna.
      - Muestra de los primeros registros.
      - Verificación de claves de cruce con defunciones.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    None
    """
    print("\n[INFO] ── Diagnóstico del DataFrame final ──")
    print(f"  → Shape         : {df.shape[0]:,} filas × {df.shape[1]} columnas")
    print(f"  → Dptos únicos  : {df['COD_DPTO'].nunique()}")
    print(f"  → Mpios únicos  : {df['COD_MPIO'].nunique():,}")
    print(f"  → CPs únicos    : {df['COD_CP'].nunique():,}")
    print()
    print("  Tipos y nulos:")
    for col in df.columns:
        nulos = df[col].isna().sum()
        print(f"    {col:<25} {str(df[col].dtype):<12} nulos: {nulos}")
    print()

    print("  Muestra (primeras 5 filas):")
    display(df.head(5))

    print()
    print("  Verificación claves de cruce con defunciones:")
    ok_dpto = df["COD_DPTO"].str.len().eq(2).all()
    ok_mpio = df["COD_MPIO"].str.len().eq(5).all()
    ok_cp   = df["COD_CP"].str.len().eq(8).all()
    print(f"    COD_DPTO → 2 dígitos en todos los registros: {'✓' if ok_dpto else '✗ REVISAR'}")
    print(f"    COD_MPIO → 5 dígitos en todos los registros: {'✓' if ok_mpio else '✗ REVISAR'}")
    print(f"    COD_CP   → 8 dígitos en todos los registros: {'✓' if ok_cp   else '✗ REVISAR'}")

### 2.9. Exportar Parquet

In [12]:
def exportar_parquet(df: pd.DataFrame,
                     ruta_salida: str,
                     compresion: str = "snappy") -> None:
    """
    Exporta el DataFrame procesado a formato Parquet.

    Parameters
    ----------
    df          : pd.DataFrame
    ruta_salida : str   Ruta completa de destino con extensión .parquet.
    compresion  : str   'snappy' (default), 'gzip' o 'brotli'.

    Returns
    -------
    None
    """
    print(f"\n[INFO] Exportando a Parquet: {ruta_salida}")
    df.to_parquet(ruta_salida, index=False, compression=compresion)
    print(f"  → Guardado | Filas: {len(df):,} | Columnas: {df.shape[1]}")

## 3. EJECUCIÓN

In [13]:
# ── Carga ─────────────────────────────────────────────────────────────────────
df_raw = cargar_hoja(RUTA_ENTRADA, HOJA)

# ── Pipeline de transformación ────────────────────────────────────────────────
df_divipola = (
    df_raw
    .pipe(filtrar_filas_validas)                        # eliminar vacíos y notas al pie
    .pipe(procesar_codigos)                             # COD_DPTO / COD_MPIO / COD_CP
    .pipe(procesar_nombres)                             # NOM_DPTO / NOM_MPIO / NOM_CP
    .pipe(procesar_area_geografica, MAPA_TIPO)          # CM/CP → etiqueta descriptiva
    .pipe(procesar_coordenadas_y_geometria)             # LONGITUD / LATITUD / WKT
    .pipe(seleccionar_y_ordenar, COLUMNAS_SALIDA)       # columnas finales en orden
)

# ── Diagnóstico ───────────────────────────────────────────────────────────────
diagnosticar(df_divipola)

# ── Exportar ──────────────────────────────────────────────────────────────────
exportar_parquet(df_divipola, RUTA_SALIDA)

print("\n[OK] Pipeline completado.")

[INFO] Cargando hoja 'Cabeceras - Centros Poblados' desde: /kaggle/input/datasets/nicolasacostaa/divipola-actualizada-2025/Listados_DIVIPOLA.xlsx
  → Filas totales (con encabezado y notas): 8,430
[INFO] Filtrando filas válidas...
  → Filas válidas : 8,420
  → Filas descartadas (vacías / notas): 10
[INFO] Procesando códigos DIVIPOLA...
  → COD_DPTO muestra : ['05', '05', '05']
  → COD_MPIO muestra : ['05001', '05001', '05001']
  → COD_CP   muestra : ['05001000', '05001001', '05001004']
  → Dptos únicos     : 33
  → Mpios únicos     : 1,122
  → CPs únicos       : 8,420
[INFO] Limpiando campos de nombre...
[INFO] Procesando campo ÁREA_GEOGRÁFICA...
  → Distribución ÁREA_GEOGRÁFICA:
      Centros Poblados y Rural Disperso         7,316
      Cabecera Municipal                        1,104
[INFO] Procesando coordenadas y generando GEOMETRIA_PUNTO_WKT...
  → LONGITUD: [-81.730363, -66.963692]  | nulos: 0
  → LATITUD : [-4.198950, 13.383969]  | nulos: 0
  → WKT generados : 8,420
  → WKT muest

,COD_DPTO,NOM_DPTO,COD_MPIO,NOM_MPIO,COD_CP,NOM_CP,ÁREA_GEOGRÁFICA,LONGITUD,LATITUD,GEOMETRIA_PUNTO_WKT
0,05,ANTIOQUIA,05001,MEDELLÍN,05001000,"MEDELLÍN, DISTRITO ESPECIAL DE CIENCIA, TECNOL...",Cabecera Municipal,-75.581775,6.246631,POINT (-75.581775 6.246631)
1,05,ANTIOQUIA,05001,MEDELLÍN,05001001,PALMITAS,Centros Poblados y Rural Disperso,-75.690573,6.343919,POINT (-75.690573 6.343919)
2,05,ANTIOQUIA,05001,MEDELLÍN,05001004,SANTA ELENA,Centros Poblados y Rural Disperso,-75.501293,6.210599,POINT (-75.501293 6.210599)
3,05,ANTIOQUIA,05001,MEDELLÍN,05001009,ALTAVISTA,Centros Poblados y Rural Disperso,-75.643706,6.221429,POINT (-75.643706 6.221429)
4,05,ANTIOQUIA,05001,MEDELLÍN,05001010,AGUAS FRÍAS,Centros Poblados y Rural Disperso,-75.633948,6.233335,POINT (-75.633948 6.233335)



  Verificación claves de cruce con defunciones:
    COD_DPTO → 2 dígitos en todos los registros: ✓
    COD_MPIO → 5 dígitos en todos los registros: ✓
    COD_CP   → 8 dígitos en todos los registros: ✓

[INFO] Exportando a Parquet: /kaggle/working/divipola_cabeceras_centros_poblados.parquet
  → Guardado | Filas: 8,420 | Columnas: 10

[OK] Pipeline completado.
